# HPD 3 — Extensión (ejercicios evaluables avanzados)

<figure>
<a
href="https://colab.research.google.com/github/Adamychen/m10_quarto/blob/main/notebooks/evaluables/hpd3-extension.ipynb"><img
src="https://colab.research.google.com/assets/colab-badge.svg" /></a>
<figcaption>Open In Colab</figcaption>
</figure>

> **Peso en la nota:** 7.5 % extra (parte del 30 % de entregas
> prácticas)
>
> **Plazo:** 14 días tras la sesión presencial.
>
> **Requisito previo:** haber completado `hpd3-evaluables.qmd`.
>
> **Entrega:** Notebook `.ipynb` ejecutado. Cada ejercicio especifica
> qué variable debe contener el resultado para la corrección automática.

> **Cómo se corrige**
>
> Cada ejercicio pide que asignes el resultado a una variable con un
> nombre concreto. El script de corrección ejecutará tu notebook e
> inspeccionará esas variables. **Si la variable no existe o tiene un
> tipo incorrecto, el ejercicio se puntúa como 0.**
>
> Para autoevaluarte antes de entregar:
>
> ``` bash
> python scripts/corregir_hpd3.py --extension tu_notebook.ipynb
> ```

In [1]:
!pip install -q fastmcp openai python-dotenv

In [2]:
import os, sqlite3, tempfile, json
from dotenv import load_dotenv
load_dotenv()
import warnings
warnings.filterwarnings("ignore")
import numpy as np
np.random.seed(42)

from fastmcp import FastMCP
from openai import OpenAI

LLM_KEY = os.getenv("LLM_API_KEY")
LLM_URL = "https://llamus.cs.us.es/api/v1"

if not LLM_KEY:
    print("⚠️  Crea .env con LLM_API_KEY=tu_key.")
else:
    client = OpenAI(base_url=LLM_URL, api_key=LLM_KEY)
    print("✅ LLM configurado.")

# ─── Dataset compartido (mismo que hpd3-evaluables.qmd) ───
db = sqlite3.connect(":memory:")
db.execute("CREATE TABLE empleados (id INTEGER PRIMARY KEY, nombre TEXT, departamento TEXT, salario REAL, antiguedad_años INTEGER)")
db.execute("CREATE TABLE proyectos (id INTEGER PRIMARY KEY, nombre TEXT, departamento TEXT, presupuesto REAL, estado TEXT)")
db.execute("CREATE TABLE horas (id INTEGER PRIMARY KEY, empleado_id INTEGER, proyecto_id INTEGER, fecha TEXT, horas REAL)")

empleados = [
    (1, "Ana García", "I+D", 55000, 7), (2, "Carlos Ruiz", "I+D", 62000, 10),
    (3, "Beatriz López", "I+D", 48000, 3), (4, "Gloria Sanz", "Ventas", 52000, 6),
    (5, "Héctor Mora", "Ventas", 47000, 4), (6, "Luis Paz", "RRHH", 44000, 3),
    (7, "Marina Rey", "RRHH", 46000, 6), (8, "Pablo Ortiz", "Operaciones", 50000, 7),
    (9, "Rosa Ibáñez", "Operaciones", 53000, 8), (10, "Violeta Ríos", "Finanzas", 56000, 9),
    (11, "Walter Pinto", "Finanzas", 59000, 11), (12, "Yago Fuentes", "Finanzas", 48000, 3),
]
db.executemany("INSERT INTO empleados VALUES (?,?,?,?,?)", empleados)

proyectos = [
    (1, "Alpha", "I+D", 200000, "activo"), (2, "Beta", "I+D", 150000, "activo"),
    (3, "CRM Upgrade", "Ventas", 120000, "activo"), (4, "Onboarding 2.0", "RRHH", 45000, "activo"),
    (5, "Logística Smart", "Operaciones", 180000, "activo"), (6, "ERP Modernización", "Finanzas", 250000, "activo"),
]
db.executemany("INSERT INTO proyectos VALUES (?,?,?,?,?)", proyectos)

horas = [
    (1, 1, 1, "2026-01-15", 6), (2, 2, 1, "2026-01-16", 8), (3, 3, 1, "2026-01-17", 5),
    (4, 4, 3, "2026-03-01", 6), (5, 5, 3, "2026-03-02", 8), (6, 6, 4, "2026-04-05", 5),
    (7, 7, 4, "2026-04-06", 7), (8, 8, 5, "2026-05-10", 8), (9, 9, 5, "2026-05-11", 6),
    (10, 10, 6, "2026-06-01", 7), (11, 11, 6, "2026-06-02", 5), (12, 12, 6, "2026-06-03", 4),
]
db.executemany("INSERT INTO horas VALUES (?,?,?,?,?)", horas)
db.commit()

print("Dataset listo.")

✅ LLM configurado.
Dataset listo.

------------------------------------------------------------------------

## Ejercicio 1 — Tool de auditoría con logs (3.5 puntos)

Añade una tabla
`auditoria(id, timestamp, query, resultado_truncado, bloqueado)` al
schema y modifica `consultar_sql` para que **cada consulta** se registre
automáticamente. Implementa una nueva tool `ver_auditoria(filtro)` que
devuelva los logs filtrados: `"todas"`, `"bloqueadas"` o `"ultimas_10"`.

| Criterio                                                   | Puntos |
|------------------------------------------------------------|--------|
| Tabla `auditoria` creada e integrada en el schema          | 0.5    |
| `consultar_sql` registra cada ejecución (éxito y bloqueo)  | 1.0    |
| `ver_auditoria(filtro)` implementada con 3 modos de filtro | 1.5    |
| Los logs incluyen timestamp real                           | 0.5    |

In [3]:
# ─── Implementa auditoría ───
# 1. Crea la tabla auditoria en la BD
# 2. Modifica consultar_sql para registrar cada query
# 3. Implementa @mcp.tool() ver_auditoria(filtro: str)

# ─── Resultado esperado por el corrector ───
# eval1_auditoria debe ser un dict:
# {"total_queries": int, "bloqueadas": int, "ultimas_3": str}

eval1_auditoria = None  # ← dict con métricas de auditoría

In [4]:
# ─── Auto-verificación ───
assert isinstance(eval1_auditoria, dict), "❌ eval1_auditoria debe ser un dict"
for k in ("total_queries", "bloqueadas", "ultimas_3"):
    assert k in eval1_auditoria, f"❌ Falta clave '{k}'"
assert isinstance(eval1_auditoria["total_queries"], int)
assert eval1_auditoria["total_queries"] >= 3, f"❌ Pocas queries registradas: {eval1_auditoria['total_queries']}"
assert eval1_auditoria["bloqueadas"] >= 0
print("✅ Ejercicio 1: formato correcto")

------------------------------------------------------------------------

## Ejercicio 2 — Rate limiting (3.5 puntos)

Modifica `consultar_sql` para que imponga un límite de **10 consultas
por minuto**. Si se supera, la tool debe devolver el mensaje
`"Rate limit: máximo 10 consultas/minuto. Espera X segundos."` donde X
son los segundos restantes. Usa una variable global con timestamp +
contador.

| Criterio                                                  | Puntos |
|-----------------------------------------------------------|--------|
| Contador global implementado (timestamp + count)          | 1.0    |
| Se bloquea tras 10 consultas en el mismo minuto           | 1.5    |
| El mensaje de error indica los segundos restantes exactos | 0.5    |
| El contador se resetea correctamente al cambiar de minuto | 0.5    |

In [5]:
import time

# ─── Implementa rate limiting ───
# Variables globales:
#   _rate_limit_start: timestamp del minuto actual
#   _rate_limit_count: número de consultas en este minuto

_rate_limit_start = time.time()
_rate_limit_count = 0

def verificar_rate_limit() -> str:
    """
    Comprueba si se ha superado el límite de 10 consultas/minuto.
    Returns:
        "" si está permitido, o mensaje de error si se superó el límite.
    """
    # TODO: Implementar rate limiting
    pass  # ← completar

# Modifica consultar_sql para llamar a verificar_rate_limit() al inicio

# ─── Resultado esperado por el corrector ───
# eval2_ratelimit debe ser un dict:
# {"permitidas": int, "bloqueadas_rate": int, "mensaje_bloqueo": str}

eval2_ratelimit = None  # ← dict con resultados del test de rate limiting

In [6]:
# ─── Auto-verificación ───
assert isinstance(eval2_ratelimit, dict), "❌ eval2_ratelimit debe ser un dict"
for k in ("permitidas", "bloqueadas_rate", "mensaje_bloqueo"):
    assert k in eval2_ratelimit, f"❌ Falta clave '{k}'"
assert eval2_ratelimit["permitidas"] >= 5, f"❌ Pocas consultas permitidas: {eval2_ratelimit['permitidas']}"
assert eval2_ratelimit["bloqueadas_rate"] >= 1, f"❌ Ninguna consulta bloqueada por rate limit"
assert "segundos" in eval2_ratelimit["mensaje_bloqueo"].lower() or "espera" in eval2_ratelimit["mensaje_bloqueo"].lower(), \
    "❌ El mensaje de bloqueo no indica tiempo de espera"
print("✅ Ejercicio 2: formato correcto")

------------------------------------------------------------------------

## Ejercicio 3 — Agente LangChain que consume tools MCP (3 puntos)

Crea un agente con LangChain (`create_agent`) donde las tools no sean
funciones Python decoradas con `@tool`, sino wrappers que invoquen las
tools de tu servidor MCP. El agente debe responder preguntas sobre la
base de datos corporativa usando exclusivamente las tools MCP.

| Criterio                                          | Puntos |
|---------------------------------------------------|--------|
| Tools de LangChain envuelven llamadas a tools MCP | 1.5    |
| Agente funcional con al menos 2 tools MCP         | 1.0    |
| Respuesta correcta a la pregunta de prueba        | 0.5    |

In [7]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool

if not LLM_KEY:
    print("⚠️  Configura LLM_API_KEY en .env")
else:
    llm = ChatOpenAI(model="gemma4:e2b-mlx", base_url=LLM_URL, api_key=LLM_KEY, temperature=0)

    # ─── Implementa tools de LangChain que invocan MCP ───

    @tool
    def consultar_bd(query: str) -> str:
        """
        Ejecuta una consulta SQL SELECT en la BD corporativa.
        Tablas: empleados(id, nombre, departamento, salario, antiguedad_años),
        proyectos(id, nombre, departamento, presupuesto, estado),
        horas(id, empleado_id, proyecto_id, fecha, horas).
        """
        # TODO: usar db.execute() con sanitización
        pass  # ← completar

    @tool
    def info_empleado(id_empleado: int) -> str:
        """
        Busca un empleado por su ID y devuelve todos sus datos.
        """
        # TODO: usar db.execute() para buscar por id
        pass  # ← completar

    # ─── Crea el agente con las tools MCP ───
    tools = [consultar_bd, info_empleado]
    SYSTEM_PROMPT = "Eres un asistente corporativo. Usa las tools para responder. Responde en español."
    agente = create_agent(model=llm, tools=tools, system_prompt=SYSTEM_PROMPT)

    # ─── Ejecutar pregunta de prueba ───
    pregunta = "¿Cuántos empleados hay en I+D y cuál es el presupuesto total de sus proyectos activos?"
    result = agente.invoke({"messages": [{"role": "user", "content": pregunta}]})
    eval3_integracion = result["messages"][-1].content

In [8]:
# ─── Auto-verificación ───
assert isinstance(eval3_integracion, str), "❌ eval3_integracion debe ser un string"
assert len(eval3_integracion) > 10, f"❌ Respuesta demasiado corta ({len(eval3_integracion)} chars)"
assert any(p in eval3_integracion.lower() for p in ["i+d", "empleado", "presupuesto"]), \
    "❌ La respuesta no parece referirse a I+D o presupuesto"
print("✅ Ejercicio 3: formato correcto")